In [1]:
import os
os.chdir("C:/TFM/tfm_env/")

from pathlib import Path
import polars as pl
import pandas as pd



In [6]:
all_data = Path("C:/TFM/tfm_env/data/02_intermediate/02.2.1 SeleccionHistoricoEstadosFinancierosUnificado/AllDataClosePrices.csv")
# all_data = Path("C:/TFM/tfm_env/data/02_intermediate/02.2.1 SeleccionHistoricoEstadosFinancierosUnificadosTodosLosSectores/AllDataClosePrices.csv")


In [7]:
df = pd.read_csv(all_data, index_col="Date", parse_dates=True)
df = df[df.ticker.notna()]
df["Current Ratio"] = df["Current Ratio"].apply(lambda x: str(x).replace("x", ""))


C:\Users\Edelmín\AppData\Local\Temp\ipykernel_4232\3327315239.py:1: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(all_data, index_col="Date", parse_dates=True)


In [8]:
df.isna().sum()

Close                                 0
year                                  0
close currency                     9850
Date.1                                0
Return on Assets %                 1506
Return on Invested Capital %      21794
Return On Equity %                22056
Normalized ROIC %                 21794
Gross Profit Margin %              8453
EBITDA Margin %                    4277
Net Income Margin %                4539
Normalized Net Income Margin %     4539
Current Ratio                         0
Total Debt / Equity               73285
% Free Cash Flow Margins            823
Total Revenues                     3538
statement currency                    0
ticker                                0
sector                                0
dtype: int64

In [9]:
(df
       .groupby("ticker", group_keys=False)
       .apply(lambda g: g.asfreq("D").ffill())
       # .droplevel(0)
       ).index.unique()


C:\Users\Edelmín\AppData\Local\Temp\ipykernel_4232\2753646971.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .apply(lambda g: g.asfreq("D").ffill())
C:\Users\Edelmín\AppData\Local\Temp\ipykernel_4232\2753646971.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.asfreq("D").ffill())


DatetimeIndex(['2005-01-03 00:00:00+00:00', '2005-01-04 00:00:00+00:00',
               '2005-01-05 00:00:00+00:00', '2005-01-06 00:00:00+00:00',
               '2005-01-07 00:00:00+00:00', '2005-01-08 00:00:00+00:00',
               '2005-01-09 00:00:00+00:00', '2005-01-10 00:00:00+00:00',
               '2005-01-11 00:00:00+00:00', '2005-01-12 00:00:00+00:00',
               ...
               '2003-12-22 00:00:00+00:00', '2003-12-23 00:00:00+00:00',
               '2003-12-24 00:00:00+00:00', '2003-12-25 00:00:00+00:00',
               '2003-12-26 00:00:00+00:00', '2003-12-27 00:00:00+00:00',
               '2003-12-28 00:00:00+00:00', '2003-12-29 00:00:00+00:00',
               '2003-12-30 00:00:00+00:00', '2003-12-31 00:00:00+00:00'],
              dtype='datetime64[ns, UTC]', name='Date', length=8275, freq=None)

In [10]:
from statistics import mean
df = (df
       .groupby("ticker")
       .apply(lambda g: g.asfreq("D").ffill())
       .droplevel(0)
       )

df = df.groupby("ticker").apply(lambda g: g.fillna(mean)).droplevel(0)
df.isna().sum()

C:\Users\Edelmín\AppData\Local\Temp\ipykernel_4232\148530481.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .apply(lambda g: g.asfreq("D").ffill())
C:\Users\Edelmín\AppData\Local\Temp\ipykernel_4232\148530481.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.asfreq("D").ffill())
C:\Users\Edelmín\AppData\Local\Temp\ipykernel_4232\148530481.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a fut

Close                             0
year                              0
close currency                    0
Date.1                            0
Return on Assets %                0
Return on Invested Capital %      0
Return On Equity %                0
Normalized ROIC %                 0
Gross Profit Margin %             0
EBITDA Margin %                   0
Net Income Margin %               0
Normalized Net Income Margin %    0
Current Ratio                     0
Total Debt / Equity               0
% Free Cash Flow Margins          0
Total Revenues                    0
statement currency                0
ticker                            0
sector                            0
dtype: int64

In [11]:

sector_dummies = pd.get_dummies(df['sector'], prefix='sector')
df = pd.concat([df, sector_dummies], axis=1)

In [87]:
sector_dummies.keys()

Index(['sector_ConsumoDiscrecional'], dtype='object')

In [ ]:
# import datetime
# from statistics import mean

# time_delta = datetime.timedelta(days=2)

# sample["mean_close_windowed_60_days"] = sample.Close.rolling( "60d").apply(lambda window: mean(window))

In [12]:
import pandas as pd

def grupos_con_fecha_min(df, group_col, fecha_min):
    """
    Comprueba si en cada grupo existe la fecha mínima dada en el índice.

    Parámetros:
        df         : DataFrame con índice de fechas.
        group_col  : Columna por la que agrupar.
        fecha_min  : Fecha mínima a comprobar (str, datetime o Timestamp).

    Retorna:
        dict {grupo: True/False} indicando si la fecha mínima está en el índice de ese grupo.
    """
    fecha_min
    resultado = {}
    for nombre_grupo, grupo in df.groupby(group_col):
        fechas_grupo = grupo.index
        resultado[nombre_grupo] = fecha_min in fechas_grupo
    return resultado


In [13]:
d = grupos_con_fecha_min(df[df.sector=="ConsumoDiscrecional"], "ticker", "2010-01-01")
drop_ticker = []
for key in d:
    if not d[key]:
        drop_ticker.append(key)
drop_ticker

[1070.0,
 1999.0,
 2282.0,
 2472.0,
 601777.0,
 '601777',
 '601799',
 '603786',
 '688819',
 '8227',
 'APTV',
 'JUBLFOOD']

In [14]:
df = df[df.ticker.apply(lambda x: x not in drop_ticker)]

In [15]:
df.to_csv("C:/TFM/tfm_env/data/02_intermediate/02.3 - InputsModels/Pruebas/input_data.csv", index_label="Date")

In [78]:
import numpy as np

dates2 = df.index
dates = df.index.unique()

d = dates[(dates >= "2009-01-01") & (dates <= "2024-12-31")]

# "2009-01-01" in dates
dates.reindex(d)

(DatetimeIndex(['2009-01-01 00:00:00+00:00', '2009-01-02 00:00:00+00:00',
                '2009-01-03 00:00:00+00:00', '2009-01-04 00:00:00+00:00',
                '2009-01-05 00:00:00+00:00', '2009-01-06 00:00:00+00:00',
                '2009-01-07 00:00:00+00:00', '2009-01-08 00:00:00+00:00',
                '2009-01-09 00:00:00+00:00', '2009-01-10 00:00:00+00:00',
                ...
                '2024-12-22 00:00:00+00:00', '2024-12-23 00:00:00+00:00',
                '2024-12-24 00:00:00+00:00', '2024-12-25 00:00:00+00:00',
                '2024-12-26 00:00:00+00:00', '2024-12-27 00:00:00+00:00',
                '2024-12-28 00:00:00+00:00', '2024-12-29 00:00:00+00:00',
                '2024-12-30 00:00:00+00:00', '2024-12-31 00:00:00+00:00'],
               dtype='datetime64[ns, UTC]', name='Date', length=5844, freq=None),
 array([1459, 1460, 1461, ..., 7300, 7301, 7302], shape=(5844,)))

# Preparación de los inputs

In [20]:
%%writefile src/prep_gnn_inputs.py
# ------------------------------------------------------------
# Pipeline para preparar snapshots (x/x_seq, grafo, y, mask) para GCN/GAT
# - Modo "paper": tiempo apilado en features por nodo -> x:[N, F=W*f_base]
# - Modo "seq"  : secuencia Wxf_base por nodo -> x_seq:[N, W, f_base]
# Grafo por fecha: top-k por correlación de retornos (ventana Wc)
# ------------------------------------------------------------

from dataclasses import dataclass
from typing import List, Optional, Dict, Tuple, Literal
import numpy as np
import pandas as pd
import logging
import torch
from torch.utils.data import Dataset
try:
    from torch_geometric.data import Data
except ImportError as e:
    raise ImportError("Instala torch_geometric para usar este dataset: pip install torch-geometric") from e

# --------------------------
# Configuración principal
# --------------------------
@dataclass
class PrepConfig:
    fund_cols: list
    
    # Ventanas
    W: int = 60              # ventana temporal de features por nodo
    Wc: int = 60             # ventana para correlación del grafo
    # Rebalanceo (puntos de decisión); si no usas cartera aún, ignora
    rebalance_every: int = 1 # 1 = cada día; 5/7/10/20 según operativa
    # Grafo
    top_k: int = 10          # vecinos por nodo en el grafo
    min_obs_corr: int = 20   # mínimo de obs para incluir en corr
    # Normalización
    winsor_q: float = 0.01   # winsorizar fundamentales
    # Inputs
    mode: Literal["paper", "seq"] = "paper"
    # Targets
    target_type: Literal["rv", "ret_next", "ret_h"] = "ret_next" # realized vol aprox, retorno 1d o H días
    horizon: int = 1         # H para ret_h o rv a H (aprox diaria)
    # Columnas
    price_col: str = "Close"
    date_col: str = "Date"
    ticker_col: str = "ticker"
    sector_col: Optional[str] = "sector"   # si no hay, será None
    # Universo mínimo
    min_history: int = 16   # mínimo de histórico por ticker para entrar al universo
    # Semilla
    seed: int = 42






# ------------------------------------------------------------
# Configuración básica del logger
# ------------------------------------------------------------

logging.basicConfig(
    filename="C:/TFM/tfm_env/data/02_intermediate/02.3 - InputsModels/app.log",              # Nombre del archivo de logs
    level=logging.INFO,              # Nivel mínimo (DEBUG, INFO, WARNING, ERROR, CRITICAL)
    format="%(asctime)s - %(levelname)s - %(message)s"  # Formato del log
)

# ------------------------------------------------------------
# Utilidades de series temporales (sin leakage)
# ------------------------------------------------------------
def set_seed(seed: int = 42):
    import random, os
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

def log_returns(prices: pd.Series) -> pd.Series:
    return np.log(prices).diff()

def ewma_vol(returns: pd.Series, halflife: int = 20) -> pd.Series:
    # Volatilidad EWMA (std) -> característica de riesgo pasada
    return returns.ewm(halflife=halflife, adjust=False).std()

def momentum(prices: pd.Series, window: int = 20) -> pd.Series:
    # Momentum simple (precio_t / precio_t-window - 1)
    return prices.pct_change(window)

def zscore_cs(df: pd.DataFrame, cols: List[str], by: str) -> pd.DataFrame:
    # z-score cross-seccional por día
    def _z(g):
        return (g[cols] - g[cols].mean()) / (g[cols].std(ddof=0) + 1e-9)
    z = df.groupby(by, group_keys=False).apply(_z)
    df_z = df.copy()
    df_z[cols] = z
    return df_z

def winsorize_series(s: pd.Series, q: float = 0.01) -> pd.Series:
    lo, hi = s.quantile(q), s.quantile(1-q)
    return s.clip(lower=lo, upper=hi)

def asof_fill_fundamentals(df_funda: pd.DataFrame, date_col: str, ticker_col: str) -> pd.DataFrame:
    """
    As-of join: se asume df_funda tiene instantáneas en ciertas fechas.
    Forward-fill por ticker para que en cada día t esté vigente el último fundamental publicado <= t.
    """
    df = df_funda.sort_values([ticker_col, date_col]).copy()
    # forward-fill por ticker
    funda_cols = [c for c in df.columns if c not in [ticker_col, date_col]]
    df[funda_cols] = df.groupby(ticker_col, group_keys=False)[funda_cols].ffill()
    return df

def realized_vol_approx(returns: pd.Series, window: int) -> pd.Series:
    # Aproximación con datos diarios: sum of squared returns en ventana (proxy de RV)
    return returns.pow(2).rolling(window).sum()



# ------------------------------------------------------------
# Construcción de features base (por ticker, por día)
# ------------------------------------------------------------
def build_feature_panel(
    df_prices: pd.DataFrame,
    # df_funda: Optional[pd.DataFrame],
    cfg: PrepConfig
) -> pd.DataFrame:
    """
    Entrada:
      df_prices: columnas [date, ticker, close, ... (p.ej. volume)]
      df_funda : columnas [date, ticker, ... ratios fundamentales ...] (opcional)
    Salida:
      panel con columnas:
      [date, ticker, sector?] + features dinámicas (ret_1d, mom_20, vol_ewma_20, ...) + fundamentales normalizados
    """
    # 1) Orden y universo
    dfp = df_prices.copy()
    dfp[cfg.date_col] = pd.to_datetime(dfp[cfg.date_col])
    dfp = dfp.sort_values([cfg.ticker_col, cfg.date_col])

    # filtra universo por histórico mínimo
    counts = dfp.groupby(cfg.ticker_col)[cfg.date_col].count()
    universe = counts[counts >= cfg.min_history].index
    excluded = counts[counts < cfg.min_history].index
    dfp = dfp[dfp[cfg.ticker_col].isin(universe)].copy()

    logging.info(f"Los siguientes tickers no fueron agregados al universo: {excluded}")

    # 2) Señales de precio sin leakage
    def feat_by_ticker(g):
        g = g.sort_values(cfg.date_col).copy()
        price = g[cfg.price_col]
        r1 = log_returns(price)
        g["ret_1d"] = r1
        g["vol_ewma_20"] = ewma_vol(r1, halflife=20)
        g["mom_20"] = momentum(price, window=20)
        g["mom_60"] = momentum(price, window=60)
        return g

    dfp = dfp.groupby(cfg.ticker_col, group_keys=False).apply(feat_by_ticker)

    # # 3) Fundamentales as-of
    # if cfg.fund_cols:
    #     dff = df_prices[cfg.fund_cols + cfg.date_col + cfg.ticker_col].copy()
    #     dff[cfg.date_col] = pd.to_datetime(dff[cfg.date_col])
    #     dff = asof_fill_fundamentals(dff, cfg.date_col, cfg.ticker_col)
    #     # merge as-of (left: precios; right: funda as-of <= date)
    #     dff = dff.sort_values([cfg.ticker_col, cfg.date_col])
    #     dfp = pd.merge_asof(
    #         dfp.sort_values(cfg.date_col),
    #         dff.sort_values(cfg.date_col),
    #         on=cfg.date_col,
    #         by=cfg.ticker_col,
    #         direction="backward"
    #     )
    # else:
    #     # si no hay fundamentales, seguimos
    #     pass

    # 4) Winsorize + normalización
    # - dinámicas (ret/mom/vol) -> z-score cross-seccional por día
    dyn_cols = [c for c in ["ret_1d", "vol_ewma_20", "mom_20", "mom_60"] if c in dfp.columns]
    if dyn_cols:
        dfp = zscore_cs(dfp, dyn_cols, by=cfg.date_col)

    # - fundamentales: winsorizar por universo y luego normalizar intrasector si hay sector
    
    
    exclude = {cfg.ticker_col, cfg.date_col, cfg.price_col, "ret_1d", "vol_ewma_20", "mom_20", "mom_60"}
    if cfg.sector_col: exclude.add(cfg.sector_col)
    funda_cols = [c for c in dfp.columns if c not in exclude]
    for c in funda_cols:
        dfp[c] = winsorize_series(dfp[c], q=cfg.winsor_q)
    if cfg.sector_col and cfg.sector_col in dfp.columns:
        # z-score por día y sector
        def _z_sector(g):
            return (g[funda_cols] - g[funda_cols].mean()) / (g[funda_cols].std(ddof=0) + 1e-9)
        dfp[funda_cols] = dfp.groupby([cfg.date_col, cfg.sector_col], group_keys=False).apply(_z_sector)
    else:
        # z-score cross-seccional por día si no hay sector
        dfp = zscore_cs(dfp, funda_cols, by=cfg.date_col)

    # 5) Elimina filas muy tempranas sin suficiente histórico para ventanas
    dfp = dfp.sort_values([cfg.ticker_col, cfg.date_col]).reset_index(drop=True)
    return dfp

# ------------------------------------------------------------
# Construcción del grafo por fecha (correlación top-k)
# ------------------------------------------------------------
def build_corr_graph_from_returns(
    ret_window: np.ndarray,  # shape [Wc, N]
    top_k: int
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    ret_window: retornos pasados hasta t, sin NaN si es posible (rellenar con 0 donde falte)
    Devuelve:
      edge_index: [2, E], edge_weight: [E]
    """
    Wc, N = ret_window.shape
    C = np.corrcoef(ret_window.T)  # [N,N]
    np.fill_diagonal(C, 0.0)
    # normaliza a [0,1]
    Cn = (C + 1.0) / 2.0
    rows, cols, weights = [], [], []
    for i in range(N):
        # top-k por fila i
        idx = np.argpartition(Cn[i], -top_k)[-top_k:]
        for j in idx:
            if Cn[i, j] > 0 and not np.isnan(Cn[i, j]):
                rows.append(i); cols.append(j); weights.append(Cn[i, j])
    edge_index = torch.tensor([rows, cols], dtype=torch.long)
    edge_weight = torch.tensor(weights, dtype=torch.float32)
    return edge_index, edge_weight

# ------------------------------------------------------------
# Creación de snapshots (x/x_seq, grafo, y)
# ------------------------------------------------------------
def make_snapshots(
    panel: pd.DataFrame,
    cfg: PrepConfig
) -> List[Data]:
    """
    panel: salida de build_feature_panel()
    Devuelve una lista de Data (PyG), uno por fecha de snapshot.
    """
    set_seed(cfg.seed)

    # Identidades y calendario
    tickers = sorted(panel[cfg.ticker_col].unique())
    t2i: Dict[str, int] = {tic: i for i, tic in enumerate(tickers)}

    dates = sorted(panel[cfg.date_col].unique())
    # construye matrices pivote para features dinámicas y fundamentales
    # Determina columnas de features base (excluye id)
    
    feat_cols = cfg.fund_cols
    
    # matrices por feature y por ticker
    # Para rapidez, creamos un diccionario {col: pivot_table date x ticker}
    pivots: Dict[str, pd.DataFrame] = {}
    for col in feat_cols + ["ret_1d"]:
        if col in panel.columns:
            piv = panel.pivot(index=cfg.date_col, columns=cfg.ticker_col, values=col).reindex(dates)
            pivots[col] = piv

    # helper para vector de máscara de elegibilidad por fecha (cotiza y tiene historia)
    def validity_mask_at_t(date_idx: int) -> np.ndarray:
        # Válido si hay datos en todas las últimas W observaciones de al menos alguna columna dinámica básica
        start = date_idx - cfg.W + 1
        if start < 0:
            return np.zeros(len(tickers), dtype=bool)
        # usamos ret_1d como base de disponibilidad
        base = pivots.get("ret_1d")
        window = base.iloc[start:date_idx+1].values  # [W, N]
        # válido si no todo NaN en la ventana
        valid = ~np.all(np.isnan(window), axis=0)
        return valid

    # lista de fechas elegibles (suficiente historia para W y Wc y target)
    min_idx = max(cfg.W - 1, cfg.Wc - 1)
    # target depende de horizon
    last_idx = len(dates) - (cfg.horizon if cfg.target_type in ["ret_h", "rv"] else 1)
    snapshot_indices = list(range(min_idx, last_idx, cfg.rebalance_every))

    data_list: List[Data] = []
    for t_idx in snapshot_indices:
        date_t = dates[t_idx]
        mask_valid = validity_mask_at_t(t_idx)

        # --- X o X_seq
        # construimos tensor [N, W, f_base] (seq) o [N, W*f_base] (paper)
        # apilamos las ventanas por cada feature de feat_cols
        start = t_idx - cfg.W + 1
        # 3D: [W, f_base, N]
        X_stack = []
        for col in feat_cols:
            mat = pivots[col].iloc[start:t_idx+1].values  # [W, N]
            X_stack.append(mat)
        X_all = np.stack(X_stack, axis=1)  # [W, f_base, N]
        X_all = np.transpose(X_all, (2, 0, 1))  # [N, W, f_base]
        # reemplaza NaN por 0 (o mejor, por 0 solo donde mask_valid True)
        X_all = np.nan_to_num(X_all, nan=0.0)

        if cfg.mode == "seq":
            x_seq = torch.tensor(X_all, dtype=torch.float32)  # [N, W, f_base]
            x_tensor = None
        else:
            # paper: aplana [W, f_base] -> [W*f_base]
            x_tensor = torch.tensor(X_all.reshape(X_all.shape[0], -1), dtype=torch.float32)  # [N, W*f_base]
            x_seq = None

        # --- Grafo por correlación (ventana Wc sobre retornos)
        rc_start = t_idx - cfg.Wc + 1
        if rc_start < 0:
            # no debería ocurrir por snapshot_indices
            continue
        ret_mat = pivots["ret_1d"].iloc[rc_start:t_idx+1].values  # [Wc, N]
        # rellena NaN con 0 solo para correlación (mejor que dropear nodos)
        ret_mat = np.nan_to_num(ret_mat, nan=0.0)
        edge_index, edge_weight = build_corr_graph_from_returns(ret_mat, top_k=cfg.top_k)

        # --- Target por nodo
        # ret_next: y_i,t = retorno de t+1 (exceso vs benchmark se puede añadir fuera)
        # ret_h   : sum de retornos próximos H días
        # rv      : sum of squared returns próximos H días (aprox diaria)
        y = np.full(len(tickers), np.nan, dtype=np.float32)

        if cfg.target_type == "ret_next":
            next_idx = t_idx + 1
            r_next = pivots["ret_1d"].iloc[next_idx].values  # [N]
            y = r_next
        elif cfg.target_type == "ret_h":
            end_idx = t_idx + cfg.horizon
            r_slice = pivots["ret_1d"].iloc[t_idx+1:end_idx+1].values  # [H, N]
            y = np.nansum(r_slice, axis=0)
        elif cfg.target_type == "rv":
            end_idx = t_idx + cfg.horizon
            r_slice = pivots["ret_1d"].iloc[t_idx+1:end_idx+1].values  # [H, N]
            y = np.nansum(np.square(r_slice), axis=0)
        else:
            raise ValueError("target_type no soportado")

        # máscara final (válido y target no NaN)
        mask_valid = mask_valid & ~np.isnan(y)
        y = np.nan_to_num(y, nan=0.0)

        # Tensores PyTorch Geometric
        pyg_kwargs = {
            "edge_index": edge_index,
            "edge_attr": edge_weight,   # guardamos pesos como edge_attr (PyG estándar)
            "y": torch.tensor(y, dtype=torch.float32),
            "mask_valid": torch.tensor(mask_valid, dtype=torch.bool),
            "date_index": torch.tensor([t_idx], dtype=torch.long)  # útil para debug
        }
        if x_tensor is not None:
            pyg_kwargs["x"] = x_tensor  # [N, W*f_base]
        if x_seq is not None:
            pyg_kwargs["x_seq"] = x_seq # [N, W, f_base]

        data = Data(**pyg_kwargs)
        data_list.append(data)

    return data_list






# # ------------------------------------------------------------
# # Ejemplo de uso rápido
# # ------------------------------------------------------------
# if __name__ == "__main__":
#     # Ejemplo sintético mínimo (sustituye por tus dataframes reales)
#     np.random.seed(0)
#     tickers = [f"TIC{i:03d}" for i in range(30)]
#     dates = pd.bdate_range("2018-01-01", "2021-12-31")  # días hábiles
#     # precios simulados
#     df_prices = []
#     for tic in tickers:
#         px = 100 * np.exp(np.cumsum(0.001 * np.random.randn(len(dates))))
#         vol = np.random.randint(1000, 5000, len(dates))
#         df_prices.append(pd.DataFrame({"date": dates, "ticker": tic, "close": px, "volume": vol}))
#     df_prices = pd.concat(df_prices, ignore_index=True)

#     # fundamentales simulados trimestrales (as-of)
#     funda_rows = []
#     for tic in tickers:
#         f_dates = pd.bdate_range("2017-12-01", "2021-12-01", freq="QS")
#         pe = 10 + 5*np.random.randn(len(f_dates))
#         roe = 0.1 + 0.05*np.random.randn(len(f_dates))
#         funda_rows.append(pd.DataFrame({"date": f_dates, "ticker": tic, "PE": pe, "ROE": roe}))
#     df_funda = pd.concat(funda_rows, ignore_index=True)

#     cfg = PrepConfig(
#         W=60, Wc=60, top_k=10, mode="paper", target_type="ret_next",
#         horizon=1, min_history=120, rebalance_every=5
#     )

#     panel = build_feature_panel(df_prices, df_funda, cfg)
#     snapshots = make_snapshots(panel, cfg)
#     print(f"Snapshots creados: {len(snapshots)}")
#     print("Ejemplo snapshot 0:")
#     d0 = snapshots[0]
#     print("x shape:" , None if "x" not in d0 else d0.x.shape)
#     print("x_seq   :", None if "x_seq" not in d0 else d0.x_seq.shape)
#     print("y shape:", d0.y.shape)
#     print("edge_index:", d0.edge_index.shape, "edge_attr:", d0.edge_attr.shape)
#     print("mask_valid true %:", float(d0.mask_valid.float().mean())*100, "%")


Overwriting src/prep_gnn_inputs.py


In [ ]:
# %%writefile src/adapter_user_df.py
# import numpy as np
# import pandas as pd

# from src.prep_gnn_inputs import (
#     PrepConfig, make_snapshots,
#     log_returns, ewma_vol, momentum, zscore_cs, winsorize_series
# )

# USER_COLS_DYNAMIC = ["ret_1d", "vol_ewma_20", "mom_20", "mom_60"]

# # Campos fundamentales que nos has pasado (ajusta si cambian nombres)
# USER_COLS_FUNDA = [
#     'Return on Assets %', 'Return on Invested Capital %', 'Return On Equity %',
#     'Normalized ROIC %', 'Gross Profit Margin %', 'EBITDA Margin %',
#     'Net Income Margin %', 'Normalized Net Income Margin %',
#     'Current Ratio', 'Total Debt / Equity', '% Free Cash Flow Margins',
#     'Total Revenues'
# ]

# # Columnas que ignoraremos
# DROP_COLS = {'year', 'close currency', 'statement currency', 'Date.1_duplicate'}

# def build_panel_from_user_df(df_raw: pd.DataFrame, cfg: PrepConfig) -> pd.DataFrame:
#     """
#     Toma tu df único con columnas:
#     ['Close','year','close currency','Date.1', <fundamentales...>, 'statement currency','ticker','sector']
#     y devuelve un panel estandarizado con:
#       [date, ticker, close, sector?, ret_1d, vol_ewma_20, mom_20, mom_60, <fundas>, <onehot_sector>...]
#     Todo normalizado sin leakage.
#     """
#     df = df_raw.copy()

#     # 1) Renombrado básico a nombres estándar
#     if 'Date.1' not in df.columns:
#         raise ValueError("No encuentro la columna 'Date.1' con la fecha.")
#     df = df.rename(columns={'Date.1': 'date', 'Close': 'close'})
#     df['date'] = pd.to_datetime(df['date'])

#     # 2) Orden y tipos
#     needed = {'date', 'ticker', 'close'}
#     if not needed.issubset(set(df.columns)):
#         missing = needed - set(df.columns)
#         raise ValueError(f"Faltan columnas necesarias: {missing}")
#     if 'sector' not in df.columns:
#         # si no hay sector, creamos uno dummy
#         df['sector'] = 'Unknown'

#     # 3) Señales dinámicas por ticker a partir de close (sin leakage)
#     def feat_by_ticker(g):
#         g = g.sort_values('date').copy()
#         price = g['close']
#         r1 = log_returns(price)                # log-returns 1d
#         g['ret_1d'] = r1
#         g['vol_ewma_20'] = ewma_vol(r1, 20)   # riesgo pasado
#         g['mom_20'] = momentum(price, 20)     # momentum 20d
#         g['mom_60'] = momentum(price, 60)     # momentum 60d
#         return g
#     df = df.groupby('ticker', group_keys=False).apply(feat_by_ticker)

#     # 4) One-hot del sector (si aún no lo hiciste)
#     sector_dummies = pd.get_dummies(df['sector'], prefix='sector')
#     df = pd.concat([df, sector_dummies], axis=1)

#     # 5) Limpiar columnas no deseadas
#     drop_existing = [c for c in DROP_COLS if c in df.columns]
#     if drop_existing:
#         df = df.drop(columns=drop_existing)

#     # 6) Winsorize & Normalización
#     # Dinámicas: z-score cross-seccional por día
#     dyn_cols = [c for c in USER_COLS_DYNAMIC if c in df.columns]
#     if dyn_cols:
#         df = zscore_cs(df, dyn_cols, by='date')

#     # Fundamentales: ya imputaste por último valor + media por ticker (ok).
#     # Aplicamos winsorize y normalizamos por día y sector (si hay)
#     funda_cols = [c for c in USER_COLS_FUNDA if c in df.columns]
#     for c in funda_cols:
#         df[c] = winsorize_series(df[c], q=0.01)

#     if 'sector' in df.columns:
#         # z-score por día y sector
#         def _z_sector(g):
#             if not funda_cols:
#                 return g[funda_cols]
#             return (g[funda_cols] - g[funda_cols].mean()) / (g[funda_cols].std(ddof=0) + 1e-9)
#         if funda_cols:
#             df[funda_cols] = df.groupby(['date', 'sector'], group_keys=False).apply(_z_sector)
#     else:
#         # fallback: por día
#         if funda_cols:
#             df = zscore_cs(df, funda_cols, by='date')

#     # 7) Devuelve panel con todo lo necesario (el resto lo hace make_snapshots)
#     # Importante: mantenemos 'date','ticker','close','sector' y todas las features
#     return df


# # --- Helper para construir snapshots directamente ---
# def build_snapshots_from_user_df(
#     df_user: pd.DataFrame,
#     cfg: PrepConfig
# ):
#     panel = build_panel_from_user_df(df_user, cfg)
#     # IMPORTANTE: para el grafo usamos ret_1d; si hay NaN residuales, make_snapshots los maneja
#     return make_snapshots(panel, cfg)


Overwriting src/adapter_user_df.py


In [99]:
df.keys()

Index(['Close', 'year', 'close currency', 'Date.1', 'Return on Assets %',
       'Return on Invested Capital %', 'Return On Equity %',
       'Normalized ROIC %', 'Gross Profit Margin %', 'EBITDA Margin %',
       'Net Income Margin %', 'Normalized Net Income Margin %',
       'Current Ratio', 'Total Debt / Equity', '% Free Cash Flow Margins',
       'Total Revenues', 'statement currency', 'ticker', 'sector',
       'sector_ConsumoDiscrecional'],
      dtype='object')

In [56]:
window_size= 60
fecha_minima = "2009-01-01"


for group in data[data.index>="2010-01-01"].reset_index().groupby("Date"):
    print(group)
    break

(Timestamp('2010-01-01 00:00:00+0000', tz='UTC'),                             Date        Close Return on Assets %  \
0      2010-01-01 00:00:00+00:00     3.118082           0.061099   
5479   2010-01-01 00:00:00+00:00     3.403525             0.0817   
10958  2010-01-01 00:00:00+00:00     3.226759           0.055184   
16437  2010-01-01 00:00:00+00:00     5.038431           0.124269   
21916  2010-01-01 00:00:00+00:00     7.123681           0.154905   
...                          ...          ...                ...   
690453 2010-01-01 00:00:00+00:00    17.959049            0.04867   
696170 2010-01-01 00:00:00+00:00    11.399893           0.071472   
701887 2010-01-01 00:00:00+00:00  1690.108398           0.145308   
707366 2010-01-01 00:00:00+00:00  1208.213623           0.036794   
713083 2010-01-01 00:00:00+00:00    37.211208           0.002881   

       Return on Invested Capital % Return On Equity % Normalized ROIC %  \
0                          0.089223             0.1514   

In [66]:
group[1]

,Date,Close,Return on Assets %,Return on Invested Capital %,Return On Equity %,Normalized ROIC %,Gross Profit Margin %,EBITDA Margin %,Net Income Margin %,Normalized Net Income Margin %,Current Ratio,Total Debt / Equity,% Free Cash Flow Margins,Total Revenues,ticker,sector_ConsumoDiscrecional
0,2010-01-01 00:00:00+00:00,3.118082,0.061099,0.089223,0.1514,0.089223,0.951968,0.156423,0.093938,0.093938,"0,83",0.715329,-0.066832,12232.679,27.0,True
5479,2010-01-01 00:00:00+00:00,3.403525,0.0817,0.148618,0.223718,0.148618,0.180588,0.145578,0.084066,0.084066,"1,37",0.669706,-0.011629,14069.225,175.0,True
10958,2010-01-01 00:00:00+00:00,3.226759,0.055184,0.116676,0.135131,0.18566,0.201824,0.066574,0.04379,0.067397,"0,96",0.096132,0.061137,25592.964183,625.0,True
16437,2010-01-01 00:00:00+00:00,5.038431,0.124269,0.192876,0.27063,0.199769,0.34406,0.23287,0.135707,0.140519,"1,57",0.467573,0.123742,1474.12503,887.0,True
21916,2010-01-01 00:00:00+00:00,7.123681,0.154905,0.232157,0.917671,0.235583,0.521055,0.220913,0.146953,0.149042,"1,67",2.126027,0.166345,14076.9,1128.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
690453,2010-01-01 00:00:00+00:00,17.959049,0.04867,0.084341,0.11071,0.084341,0.302342,0.090118,0.023296,0.023296,"2,36",0.432901,-0.032877,1084.646,ULTA,True
696170,2010-01-01 00:00:00+00:00,11.399893,0.071472,0.106567,0.125158,0.124174,0.442526,0.126176,0.063885,0.075565,"2,40",0.311161,0.121587,7220.286,VFC,True
701887,2010-01-01 00:00:00+00:00,1690.108398,0.145308,0.260569,0.391839,0.289163,0.332322,0.100543,0.053777,0.054204,"1,28",0.457892,0.033557,23393.0,WHL,True
707366,2010-01-01 00:00:00+00:00,1208.213623,0.036794,0.062411,0.075092,0.07434,0.855387,0.245317,0.068785,0.083998,"0,44",0.591949,-0.048254,1334.6,WTB,True


In [36]:
d

,Close,Return on Assets %,Return on Invested Capital %,Return On Equity %,Normalized ROIC %,Gross Profit Margin %,EBITDA Margin %,Net Income Margin %,Normalized Net Income Margin %,Current Ratio,Total Debt / Equity,% Free Cash Flow Margins,Total Revenues,ticker,sector_ConsumoDiscrecional
Date,,,,,,,,,,,,,,,
2003-01-01 00:00:00+00:00,0.843766,0.010031,0.025073,0.017685,0.026581,<function mean at 0x000001A4A361E0C0>,<function mean at 0x000001A4A361E0C0>,<function mean at 0x000001A4A361E0C0>,<function mean at 0x000001A4A361E0C0>,"664,47",0.78261,5494000.0,<function mean at 0x000001A4A361E0C0>,PMV,True


In [4]:
input_data_paht = "C:/TFM/tfm_env/data/02_intermediate/02.3 - InputsModels/Pruebas/input_data.csv"
df = pd.read_csv(input_data_paht, index_col="Date", parse_dates=True)


C:\Users\Edelmín\AppData\Local\Temp\ipykernel_4232\231416372.py:2: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_data_paht, index_col="Date", parse_dates=True)


In [23]:
data = df[['Close', 'Return on Assets %',
       'Return on Invested Capital %', 'Return On Equity %',
       'Normalized ROIC %', 'Gross Profit Margin %', 'EBITDA Margin %',
       'Net Income Margin %', 'Normalized Net Income Margin %',
       'Current Ratio', 'Total Debt / Equity', '% Free Cash Flow Margins',
       'Total Revenues', 'ticker', 'sector_ConsumoDiscrecional']]

In [24]:
from src.prep_gnn_inputs import PrepConfig, make_snapshots
# from src.adapter_user_df import build_snapshots_from_user_df

cfg = PrepConfig(
    W=60,           # ventana temporal de features por nodo
    Wc=60,          # ventana para la correlación del grafo
    top_k=10,       # vecinos en el grafo
    mode="paper",   # "paper" -> x:[N, W*f_base];  "seq" -> x_seq:[N,W,f_base]
    target_type="ret_next",  # etiqueta: retorno de t+1
    horizon=1,
    rebalance_every=5,  # snapshots cada 5 días (tu operativa)
    min_history=120,     # exige historia mínima por ticker
    fund_cols = [
    'Return on Assets %', 'Return on Invested Capital %', 'Return On Equity %',
    'Normalized ROIC %', 'Gross Profit Margin %', 'EBITDA Margin %',
    'Net Income Margin %', 'Normalized Net Income Margin %',
    'Current Ratio', 'Total Debt / Equity', '% Free Cash Flow Margins',
    'Total Revenues', 'sector_ConsumoDiscrecional'
]
)

# df_user: tu dataframe con las columnas listadas
snapshots = make_snapshots(data, cfg)

print(f"Snapshots: {len(snapshots)}")
d0 = snapshots[0]
print("x:", None if "x" not in d0 else d0.x.shape)
print("x_seq:", None if "x_seq" not in d0 else d0.x_seq.shape)
print("y:", d0.y.shape, "edge_index:", d0.edge_index.shape, "edge_attr:", d0.edge_attr.shape)
print("mask_valid %:", float(d0.mask_valid.float().mean())*100)


TypeError: '<' not supported between instances of 'str' and 'float'